In [ ]:
#Bibliotecas

import pandas as pd

In [ ]:
# Montar o Google Drive no Colab
#from google.colab import drive
#drive.mount('/content/drive')


In [ ]:
# Caminho do arquivo no Google Drive
caminho = "data/censo_ed_basica_2023_reduzido.csv"

# Lê a base original
df = pd.read_csv(caminho, sep=",", encoding="utf-8")

In [ ]:
colunas_manter = [
    # 1. Identificação e chaves
    "NU_ANO_CENSO", "CO_MUNICIPIO", "NO_MUNICIPIO",
    "SG_UF", "NO_UF",
    "CO_ENTIDADE", "NO_ENTIDADE",

    # 2. Contexto da escola
    "TP_DEPENDENCIA", "TP_LOCALIZACAO", "TP_SITUACAO_FUNCIONAMENTO",

    # 3. Educação Especial
    "IN_ESP", "IN_ESP_CC", "IN_ESP_CE",
    "QT_MAT_ESP", "QT_MAT_ESP_CC", "QT_MAT_ESP_CE",

    # 4. Infraestrutura / inclusão
    "IN_BIBLIOTECA",
    "IN_LABORATORIO_INFORMATICA",
    "IN_QUADRA_ESPORTES", "IN_QUADRA_ESPORTES_COBERTA", "IN_QUADRA_ESPORTES_DESCOBERTA",
    "IN_INTERNET", "IN_INTERNET_ALUNOS", "IN_BANDA_LARGA",
    "IN_ACESSIBILIDADE_RAMPAS", "IN_ACESSIBILIDADE_ELEVADOR",
    "IN_ACESSIBILIDADE_PISOS_TATEIS", "IN_ACESSIBILIDADE_SINAL_VISUAL", "IN_ACESSIBILIDADE_SINAL_TATIL",

    # 5. Perfil dos alunos
    "QT_MAT_BAS", "QT_MAT_BAS_FEM", "QT_MAT_BAS_MASC",
    "QT_MAT_BAS_BRANCA", "QT_MAT_BAS_PRETA", "QT_MAT_BAS_PARDA", "QT_MAT_BAS_INDIGENA",
    "QT_MAT_BAS_6_10", "QT_MAT_BAS_11_14", "QT_MAT_BAS_15_17",

    # 6. Professores e RH
    "QT_DOC_BAS", "QT_DOC_FUND", "QT_DOC_MED",
    "IN_PROF_PSICOLOGO", "IN_PROF_ASSIST_SOCIAL", "IN_PROF_FONAUDIOLOGO"
]


In [ ]:
# Mantém só as colunas filtradas
df_reduzido = df[colunas_manter]

# Visualização interativa
from google.colab import data_table
data_table.enable_dataframe_formatter()

df_reduzido.head(50)

In [ ]:
#limpar espaços extras

df_reduzido.columns = df_reduzido.columns.str.strip()

In [ ]:
# Verificando quantos campos númos existem em cada coluna

df[colunas_manter].isnull().sum()  # mostra quantos NaN existem em cada coluna


In [ ]:
# Alterar colunas núméricas com Nan (valor nulo) por 0

colunas_numericas = [
    "QT_MAT_ESP", "QT_MAT_ESP_CC", "QT_MAT_ESP_CE",
    "QT_MAT_BAS", "QT_MAT_BAS_FEM", "QT_MAT_BAS_MASC",
    "QT_MAT_BAS_BRANCA", "QT_MAT_BAS_PRETA", "QT_MAT_BAS_PARDA", "QT_MAT_BAS_INDIGENA",
    "QT_MAT_BAS_6_10", "QT_MAT_BAS_11_14", "QT_MAT_BAS_15_17",
    "QT_DOC_BAS", "QT_DOC_FUND", "QT_DOC_MED"
]

df_reduzido[colunas_numericas] = df_reduzido[colunas_numericas].fillna(0)


In [ ]:
# Alterar colunas categóricas com Nan (valor nulo) por Desconhecido

colunas_categoricas = [
    "IN_ESP", "IN_ESP_CC", "IN_ESP_CE",
    "IN_BIBLIOTECA", "IN_LABORATORIO_INFORMATICA",
    "IN_QUADRA_ESPORTES", "IN_QUADRA_ESPORTES_COBERTA", "IN_QUADRA_ESPORTES_DESCOBERTA",
    "IN_INTERNET", "IN_INTERNET_ALUNOS", "IN_BANDA_LARGA",
    "IN_ACESSIBILIDADE_RAMPAS", "IN_ACESSIBILIDADE_ELEVADOR",
    "IN_ACESSIBILIDADE_PISOS_TATEIS", "IN_ACESSIBILIDADE_SINAL_VISUAL", "IN_ACESSIBILIDADE_SINAL_TATIL",
    "IN_PROF_PSICOLOGO", "IN_PROF_ASSIST_SOCIAL", "IN_PROF_FONAUDIOLOGO"
]

df_reduzido[colunas_categoricas] = df_reduzido[colunas_categoricas].fillna("Desconhecido")


In [ ]:
# Remover possíveis duplicatas atrasvés do código das escolas

df_reduzido = df_reduzido.drop_duplicates(subset=["CO_ENTIDADE"])


In [ ]:
# Verificação de consistência entre colunas

df_reduzido["check_total"] = df_reduzido["QT_MAT_BAS_FEM"] + df_reduzido["QT_MAT_BAS_MASC"]
inconsistentes = df_reduzido[df_reduzido["check_total"] != df_reduzido["QT_MAT_BAS"]]


In [ ]:
print(inconsistentes.shape)   # mostra quantas linhas estão incoerentes
inconsistentes.head(10)       # mostra os primeiros casos


In [ ]:
# Salvar em CSV no Google Drive

df_reduzido.to_csv('results/censo_2023_tratado.csv', index=False)


# O que foi feito nessa etapa?
Realizado o tratamento dos dados do Censo de 2023 (dados já reduzidos anteriormente a 10.000 linhas), retiramos tabelas que acreditamos não ser proveitosas para nossa pesquisa e alteramos valores nulos.

*   Em relação a colunas numéricas, na base do Censo, NaN geralmente significa não há alunos/professores naquela categoria. Então reitrar evita problemas em cálculos estatísticos, médias, somas, ou análises de correlação.
*   Já em relação a colunas categóricas, esses valores são qualitativos. Não sabemos se o serviço existe ou não, marcar como desconhecido preserva a informação sem assumir nada que atrapalhe a analise futura.

Além disso, utilizamos o comando df.isnull().sum() para verificar a quantidade de valores nulos (NaN) em cada coluna da base de dados. O objetivo é identificar quais colunas estavam incompletas ou com informações ausentes, o que poderia comprometer a análise.
Realizado a verificação de consistência entre colunas, criamos uma nova coluna chamada check_total, que soma os valores de matrículas feminina (QT_MAT_BAS_FEM) e masculinas (QT_MAT_BAS_MASC). Depois, comparamos essa soma com a coluna QT_MAT_BAS, que representa o total de matrículas informado na base. As linhas em que os valores não batem foram armazenadas em um novo dataframe chamado inconsistentes.
O objetivo era verificar se a soma das matrículas femininas e masculinas realmente corresponde ao total informado. Essa é uma checagem de consistência dos dados, para identificar erros de digitação, falhas de preenchimento ou incoerências na base.